<a href="https://colab.research.google.com/github/sunitharamu-1983/GENAI-Learning-Journey/blob/main/Transformer_Build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/02_Transformer_Architecture/notebooks/01_transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer from Scratch

This notebook breaks down the complex Transformer model into understandable, bite-sized pieces. We'll go through each building block, how they connect, and finally, how the entire model learns to translate languages.

**Credit:** This notebook is adapted from the original work by [AI Maker Space](https://github.com/AI-Maker-Space/).


## Major components of transformers

1. The Building Blocks
2. Training the Model

## Reference

[Attention is All You Need](https://arxiv.org/pdf/1706.03762.pdf).

## Libraries

Leveraging the [PyTorch]() library

## Dataset for Training
Toy Dataset with limited data, Epochs, Seq length, Layers and Batch size to make the best use of memory.

### 1. Input Embeddings: Turning Words into Numbers

Computers don't understand words directly. They need numbers! This layer takes each word in our sentence and converts it into a special numerical 'vector' or 'embedding'. Words that mean similar things will have similar numerical representations. It's like giving each word a unique, meaningful ID badge. This component also scales the embeddings by the square root of the model's dimension (`d_model`), which helps stabilize training.

In [ ]:
import torch
import torch.nn as nn
import math

class InputEmbeddings(nn.Module):
  def __init__(self, d_model: int, vocab_size: int, verbose=False) -> None:
    """
    vocab_size - the size of our vocabulary
    d_model - the dimension of our embeddings and the input dimension for our model
    """
    super().__init__()
    self.vocab_size = vocab_size
    self.d_model = d_model
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.verbose = verbose

  def forward(self, x):
    if self.verbose:
      print(f"Embedding Vector (1st 5 elements): {self.embedding(x)[:5] * math.sqrt(self.d_model)}")
    return self.embedding(x) * math.sqrt(self.d_model) # scale embeddings by square root of d_model

### 2. Positional Encoding: Understanding Word Order

Unlike traditional language models that process words one by one, Transformers look at all words in a sentence at once. This means they lose the crucial information about word order. Positional Encoding adds a special 'hint' to each word's numerical representation, telling the model where it is in the sentence (e.g., 'first word', 'second word', etc.).

In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model: int, seq_len: int, dropout: float, verbose=False) -> None:
    super().__init__()
    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)
    self.verbose=verbose

    positional_embeddings = torch.zeros(seq_len, d_model)
    positional_sequence_vector = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    positional_model_vector = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    positional_embeddings[:, 0::2] = torch.sin(positional_sequence_vector * positional_model_vector)
    positional_embeddings[:, 1::2] = torch.cos(positional_sequence_vector * positional_model_vector)
    positional_embeddings = positional_embeddings.unsqueeze(0)

    self.register_buffer('positional_embeddings', positional_embeddings)

  def forward(self, x):
    x = x + (self.positional_embeddings[:, :x.shape[1], :]).requires_grad_(False)
    if self.verbose:
      print(f"Positional Encodings (1st 5 elements): {x}")
    return self.dropout(x)

### 3. Layer Normalization: Keeping Things Stable

Imagine you're trying to balance many different scales at once. Layer Normalization helps keep the numerical values within the model stable and prevents them from becoming too large or too small. This makes the training process smoother and faster. It essentially applies a normalization step across the features for each sample independently, helping to maintain a consistent scale for the activations.

In [ ]:
class LayerNormalization(nn.Module):
  def __init__(self, features: int, epsilon:float=10**-6) -> None:
    super().__init__()
    self.epsilon = epsilon
    # gamma is a learnable parameter, for scaling
    self.gamma = nn.Parameter(torch.ones(features))
    # beta is a learnable parameter, for shifting
    self.beta = nn.Parameter(torch.zeros(features))

  def forward(self, x):
    # Calculate mean and standard deviation across the last dimension (features)
    mean = x.mean(dim = -1, keepdim = True)
    standard_deviation = x.std(dim = -1, keepdim = True)
    # Normalize the input, apply gamma and beta
    return self.gamma * (x - mean) / (standard_deviation + self.epsilon) + self.beta

### 4. Residual Connection: Shortcut for Learning

This is like adding a shortcut or a direct path for information to flow through the network. It helps prevent a problem called 'vanishing gradients,' which can make it hard for deep neural networks to learn effectively. It essentially says, 'let's just pass the original information through, and then add any new insights on top of it.'

In [ ]:
class ResidualConnection(nn.Module):
  def __init__(self, features: int, dropout: float = 0.1) -> None:
    super().__init__()
    self.dropout = nn.Dropout(dropout)
    self.layernorm = LayerNormalization(features)

  def forward(self, x, sublayer):
    # The residual connection adds the input 'x' to the output of the sublayer.
    # The sublayer's output is first normalized and then passed through a dropout layer.
    return x + self.dropout(sublayer(self.layernorm(x)))

### 5. Feed Forward Network: Adding Complexity and Depth

This is a standard neural network layer that processes the information further. It helps the model learn more complex patterns and relationships within the language data. It's applied to each word's representation independently, allowing the model to think deeply about each word's context.

In [ ]:
class FeedForwardBlock(nn.Module):
  def __init__(self, d_model: int, d_ff: int = 2048, dropout: float = 0.1) -> None:
    """
    d_model - dimension of model (input and output dimension)
    d_ff - dimension of the inner-layer of the feed forward network
    dropout - regularization measure
    """
    super().__init__()
    self.linear_1 = nn.Linear(d_model, d_ff) # W1 and B1
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff, d_model) # W2 and B2

  def forward(self, x):
    # (Batch, Seq Len, d_model) -> (Batch, Seq Len, d_ff) -> (Batch, Seq Len, d_model)
    return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

### 6. Multi-Head Attention: Focusing on Important Words

This is the *heart* of the Transformer. Imagine you're reading a sentence, and for each word, you want to pay attention to other relevant words in the sentence. Multi-Head Attention does just that. It allows the model to simultaneously look at different parts of the input sentence, giving varying levels of importance (attention) to different words when processing a particular word.

`Q`, `K`, `V` stand for:
*   **Query (Q):** What I'm looking for.
*   **Key (K):** What you have to offer.
*   **Value (V):** What you actually offer.

Essentially, for each word (Query), the model compares it to all other words (Keys) to figure out how relevant they are. Then, it uses those relevance scores to weigh and combine the information (Values) from those important words.

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model: int = 512, num_heads: int = 8, dropout: float = 0.1) -> None:
    super().__init__()
    self.d_model = d_model # Dimension of the model (input/output features)
    self.num_heads = num_heads # Number of attention heads
    assert d_model % num_heads == 0, "d_model is not divisible by num_heads"

    self.d_k = d_model // num_heads # Dimension of Query, Key, Value for each head

    # Linear layers to project input into Query, Key, Value for all heads
    self.w_q = nn.Linear(d_model, d_model, bias=False) # W_Q
    self.w_k = nn.Linear(d_model, d_model, bias=False) # W_K
    self.w_v = nn.Linear(d_model, d_model, bias=False) # W_V

    # Linear layer to combine outputs from all heads
    self.w_o = nn.Linear(d_model, d_model, bias=False) # W_O

    self.dropout = nn.Dropout(dropout)

  @staticmethod
  def attention(query, key, value, mask, dropout: nn.Dropout = None):
    d_k = query.shape[-1] # Get the dimension of K (d_k)

    # Calculate attention scores: (Query @ Key.T) / sqrt(d_k)
    attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

    # Apply mask if provided (e.g., for padding or causality)
    if mask is not None:
      attention_scores.masked_fill_(mask == 0, -1e9) # Fill masked positions with a very small number

    # Apply softmax to get attention probabilities
    attention_scores = attention_scores.softmax(dim=-1)

    # Apply dropout to attention scores
    if dropout is not None:
      attention_scores = dropout(attention_scores)

    # Multiply attention probabilities by Value to get the weighted sum
    return (attention_scores @ value), attention_scores

  def forward(self, query, key, value, mask):
    # Apply linear transformations to Query, Key, Value
    query = self.w_q(query)
    key = self.w_k(key)
    value = self.w_v(value)

    # Split into multiple heads and transpose dimensions for parallel processing
    # (Batch, Seq Len, d_model) -> (Batch, Seq Len, num_heads, d_k) -> (Batch, num_heads, Seq Len, d_k)
    query = query.view(query.shape[0], query.shape[1], self.num_heads, self.d_k).transpose(1, 2)
    key = key.view(key.shape[0], key.shape[1], self.num_heads, self.d_k).transpose(1, 2)
    value = value.view(value.shape[0], value.shape[1], self.num_heads, self.d_k).transpose(1, 2)

    # Perform scaled dot-product attention
    x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)

    # Concatenate heads and apply final linear transformation
    # (Batch, num_heads, Seq Len, d_k) -> (Batch, Seq Len, num_heads, d_k) -> (Batch, Seq Len, d_model)
    x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.num_heads * self.d_k)

    # Apply final linear layer to project back to d_model
    return self.w_o(x)

## Part 2: Assembling the Encoder and Decoder (The Two Main Sections of the Brain)

The Transformer architecture is split into two main parts: the **Encoder** and the **Decoder**. The Encoder understands the input sentence, and the Decoder generates the output sentence.

### The Encoder Block: Understanding the Input

The Encoder Block takes our word embeddings with positional information and processes them. It uses the Multi-Head Attention to understand how each word relates to other words in the *input* sentence. Then, a Feed Forward Network adds more processing depth. This process is repeated multiple times (in an 'Encoder Stack') to build a rich understanding of the entire input sentence.

In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self, features: int, self_attention_block: MultiHeadAttention, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
    super().__init__()
    self.self_attention_block = self_attention_block
    self.feed_forward_block = feed_forward_block
    # An encoder block has two residual connections: one for self-attention and one for the feed-forward network
    self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

  def forward(self, x, input_mask):
    # First residual connection: Apply self-attention
    x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, input_mask))
    # Second residual connection: Apply feed-forward network
    x = self.residual_connections[1](x, self.feed_forward_block)
    return x

### The Encoder Stack: Multiple Layers of Understanding

We don't just use one Encoder Block; we stack several of them (typically 6 in the original paper, but fewer in our small example). Each layer refines the understanding of the input sentence, passing its refined output to the next layer.

In [ ]:
class EncoderStack(nn.Module):
  def __init__(self, features: int, layers: nn.ModuleList) -> None:
    super().__init__()
    self.layers = layers # A list of EncoderBlock instances
    self.norm = LayerNormalization(features) # Final Layer Normalization after all blocks

  def forward(self, x, mask):
    for layer in self.layers:
      x = layer(x, mask) # Pass through each EncoderBlock sequentially
    return self.norm(x)

### The Decoder Block: Generating the Output

The Decoder Block is responsible for generating the output sentence (e.g., the Italian translation). It has three main parts:

1.  **Self-Attention (Masked):** It pays attention to the words it has *already generated* in the output sentence. The 'masked' part means it can only look at previous words, not future ones, preventing it from 'cheating'.
2.  **Cross-Attention:** This is where the magic happens for translation! The decoder looks at the *encoder's output* (its understanding of the input sentence) and identifies which parts of the input are most relevant for generating the *next word* in the output sentence.
3.  **Feed Forward Network:** Similar to the encoder, this adds more processing power.

In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self, features: int, self_attention_block: MultiHeadAttention, cross_attention_block: MultiHeadAttention, feed_forward_block: FeedForwardBlock, dropout: float) -> None:
    super().__init__()
    self.self_attention_block = self_attention_block
    self.cross_attention_block = cross_attention_block
    self.feed_forward_block = feed_forward_block
    # A decoder block has three residual connections:
    # one for masked self-attention, one for cross-attention, and one for the feed-forward network
    self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

  def forward(self, x, encoder_output, input_mask, target_mask):
    # First residual connection: Masked self-attention (decoding)
    x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, target_mask))
    # Second residual connection: Cross-attention (connecting encoder and decoder)
    x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, input_mask))
    # Third residual connection: Feed-forward network
    x = self.residual_connections[2](x, self.feed_forward_block)
    return x

### The Decoder Stack: Layers of Generation

Similar to the encoder, we stack multiple Decoder Blocks. Each layer refines the generated output and combines it with the encoder's understanding to produce better translations. **6 LAYERS similar to encoder stack**

In [ ]:
class DecoderStack(nn.Module):
  def __init__(self, features: int, layers: nn.ModuleList) -> None:
    super().__init__()
    self.layers = layers
    self.norm = LayerNormalization(features)

  def forward(self, x, encoder_output, input_mask, target_mask):
    for layer in self.layers:
      x = layer(x, encoder_output, input_mask, target_mask)
    return self.norm(x)

## Linear Projection Layer

After the decoder's self-attention and encoder-decoder attention layers, we have a context vector representing each Italian word. This context vector has a high dimension (e.g. 512 or 1024).

We want to take this context vector and generate a probability distribution over the French vocabulary so we can pick the next translated word.

The linear projection layer helps with this. It projects the context vector into a much larger vector called the vocabulary distribution - one entry per word in the vocabulary.

For example, if our Italian vocabulary has 50,000 words, the vocabulary distribution will have 50,000 dimensions. Each dimension corresponds to the probability of that Italian word being the correct translation.

In [ ]:
class LinearProjectionLayer(nn.Module):
  def __init__(self, d_model, vocab_size) -> None:
    super().__init__()
    self.proj = nn.Linear(d_model, vocab_size)

  def forward(self, x) -> None:
    return self.proj(x)

## The Transformer

At this point, all we need to do is create a class that represents our model!

In [ ]:
class Transformer(nn.Module):
  def __init__(self, encoder: EncoderStack, decoder: DecoderStack, src_embed: InputEmbeddings, tgt_embed: InputEmbeddings, src_pos: PositionalEncoding, tgt_pos: PositionalEncoding, projection_layer: LinearProjectionLayer) -> None:
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.src_pos = src_pos
    self.tgt_pos = tgt_pos
    self.projection_layer = projection_layer

  def encode(self, src, src_mask):
    src = self.src_embed(src)
    src = self.src_pos(src)
    return self.encoder(src, src_mask)

  def decode(self, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt: torch.Tensor, tgt_mask: torch.Tensor):
    tgt = self.tgt_embed(tgt)
    tgt = self.tgt_pos(tgt)
    return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

  def project(self, x):
    return self.projection_layer(x)

## Building Our Transformer

Now that we have each of our components - we need to construct an actual model!

We'll use this helper function to aid in our goal and set up our Encoder/Decoder Stacks!

In [ ]:
def build_transformer(input_vocab_size: int, target_vocab_size: int, input_seq_len: int, target_seq_len: int, d_model: int=512, N: int=6, num_heads: int=8, dropout: float=0.1, d_ff: int=2048, verbose=True) -> Transformer:
  input_embeddings = InputEmbeddings(d_model, input_vocab_size, verbose=verbose)
  target_embeddings = InputEmbeddings(d_model, target_vocab_size)

  input_position = PositionalEncoding(d_model, input_seq_len, dropout, verbose=verbose)
  target_position = PositionalEncoding(d_model, target_seq_len, dropout)

  encoder_blocks = []

  for _ in range(N):
    encoder_self_attention_block = MultiHeadAttention(d_model, num_heads, dropout)
    feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
    encoder_block = EncoderBlock(d_model, encoder_self_attention_block, feed_forward_block, dropout)
    encoder_blocks.append(encoder_block)

  decoder_blocks = []

  for _ in range(N):
    decoder_self_attention_block = MultiHeadAttention(d_model, num_heads, dropout)
    decoder_cross_attention_block = MultiHeadAttention(d_model, num_heads, dropout)
    feed_forward_block = FeedForwardBlock(d_model, d_ff, dropout)
    decoder_block = DecoderBlock(d_model, decoder_self_attention_block, decoder_cross_attention_block, feed_forward_block, dropout)
    decoder_blocks.append(decoder_block)

  encoder_stack = EncoderStack(d_model, nn.ModuleList(encoder_blocks))
  decoder_stack = DecoderStack(d_model, nn.ModuleList(decoder_blocks))

  linear_projection_layer = LinearProjectionLayer(d_model, target_vocab_size)

  transformer = Transformer(encoder_stack, decoder_stack, input_embeddings, target_embeddings, input_position, target_position, linear_projection_layer)

  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

## Part 4: Training the Transformer (Teaching the Brain)

Once we have our Transformer model, it's like a newborn brain – it doesn't know anything yet. We need to train it using a lot of example sentences. This involves providing it with English sentences and their Italian translations, and it learns to map one to the other.

### Dataset Creation: Preparing the Learning Material

This `BilingualDataset` class is like preparing flashcards for our model. It takes raw sentences, converts them into numerical tokens, adds special 'start' and 'end' markers, and pads shorter sentences so they all have the same length. It also creates 'masks' to help the attention mechanism focus correctly (e.g., preventing the decoder from looking at future words).

## Dataset Creation

The BilingualDataset is a custom PyTorch dataset for working with translation data. It needs a tokenizer for each language, a dataset of sentence pairs, info on which languages are source and target, and the max sequence length.

This class handles tokenizing the sentences, padding them to be the same length, and getting the data into the right format for sequence-to-sequence models. It adds special start, end, and padding tokens so all the inputs and outputs are the same length.

When you grab a sample from the dataset, it tokenizes the source and target sentences, pads them, and creates the input tensors the model needs - encoder input, decoder input, and target labels. It also makes masks to show what's real data vs padding, and to make sure the decoder predictions only use previous tokens, not future ones.

The BilingualDataset gets the data ready for training seq2seq models in a way that works with the sequential nature of language. The model can only predict the next token based on what came before it, not after.

In [ ]:
from torch.utils.data import Dataset

class BilingualDataset(Dataset):
  def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
    super().__init__()
    self.seq_len = seq_len

    self.ds = ds
    self.tokenizer_src = tokenizer_src
    self.tokenizer_tgt = tokenizer_tgt
    self.src_lang = src_lang
    self.tgt_lang = tgt_lang

    self.sos_token = torch.tensor([tokenizer_tgt.token_to_id("[SOS]")], dtype=torch.int64)
    self.eos_token = torch.tensor([tokenizer_tgt.token_to_id("[EOS]")], dtype=torch.int64)
    self.pad_token = torch.tensor([tokenizer_tgt.token_to_id("[PAD]")], dtype=torch.int64)

  def __len__(self):
    return len(self.ds)

  def __getitem__(self, idx):
    src_target_pair = self.ds[idx]
    src_text = src_target_pair['translation'][self.src_lang]
    tgt_text = src_target_pair['translation'][self.tgt_lang]

    enc_input_tokens = self.tokenizer_src.encode(src_text).ids
    dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

    enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
    dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1

    if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
        raise ValueError("Sentence is too long")

    encoder_input = torch.cat(
        [
            self.sos_token,
            torch.tensor(enc_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.int64),
        ],
        dim=0,
    )

    decoder_input = torch.cat(
        [
            self.sos_token,
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64),
        ],
        dim=0,
    )

    label = torch.cat(
        [
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.int64),
        ],
        dim=0,
    )

    assert encoder_input.size(0) == self.seq_len
    assert decoder_input.size(0) == self.seq_len
    assert label.size(0) == self.seq_len

    return {
        "encoder_input": encoder_input,
        "decoder_input": decoder_input,
        "encoder_mask": (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(),
        "decoder_mask": (decoder_input != self.pad_token).unsqueeze(0).int() & causal_mask(decoder_input.size(0)),
        "label": label,
        "src_text": src_text,
        "tgt_text": tgt_text,
    }

def causal_mask(size):
  mask = torch.triu(torch.ones((1, size, size)), diagonal=1).type(torch.int)
  return mask == 0

### Tokenizer Builder: Teaching the Model to Read and Write

A tokenizer breaks down sentences into individual 'tokens' (words or sub-words) and assigns a unique numerical ID to each. This function builds a tokenizer for both our source (English) and target (Italian) languages, which is essential for converting text to numbers and vice-versa.

In [ ]:
!pip install transformers tokenizers datasets -qU

def get_all_sentences(ds, lang):
    for item in ds:
        yield item['translation'][lang]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00


We'll quickly train a tokenizer on our dataset for both our source and target languages.

We'll be sure to add the `[UNK]`, `[PAD]`, `[SOS]`, and `[EOS]` special tokens.

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

def build_tokenizer(config, ds, lang):
  tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
  tokenizer.pre_tokenizer = Whitespace()
  trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
  tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
  return tokenizer

### `get_ds` Function: Getting the Full Training Setup

This function orchestrates the data preparation. It loads the raw dataset, builds the tokenizers, filters sentences that are too long or too short for our model, splits the data into training and validation sets, and creates data loaders that feed batches of data to the model during training.

Now we can create our dataset in a format that our model expects and can train with!

In [ ]:
from torch.utils.data import DataLoader, random_split

def get_ds(config):
  # It only has the train split, so we divide it overselves
  ds_raw = load_dataset(f"{config['datasource']}", f"{config['lang_src']}-{config['lang_tgt']}", split='train')

  # Build tokenizers
  tokenizer_src = build_tokenizer(config, ds_raw, config['lang_src'])
  tokenizer_tgt = build_tokenizer(config, ds_raw, config['lang_tgt'])

  # Filter sentences based on seq_len constraints
  filtered_data = []
  max_seq_len = config['seq_len']
  min_raw_tokens = 10 # Heuristic: sentences with less than 10 raw tokens are considered too short
  removed_count = 0

  # First pass to filter data and find true max lengths of filtered data
  for item in ds_raw:
    src_text = item['translation'][config['lang_src']]
    tgt_text = item['translation'][config['lang_tgt']]

    src_ids = tokenizer_src.encode(src_text).ids
    tgt_ids = tokenizer_tgt.encode(tgt_text).ids

    # Check for minimum length (raw tokens)
    is_too_short = len(src_ids) < min_raw_tokens or len(tgt_ids) < min_raw_tokens

    # Check if sentence is too long to fit into the sequence length after adding special tokens
    # Encoder input: raw tokens + SOS + EOS => len(src_ids) + 2
    # Decoder input: raw tokens + SOS => len(tgt_ids) + 1 (label also requires EOS, so it's len(tgt_ids) + 1)
    is_too_long = (len(src_ids) > max_seq_len - 2) or (len(tgt_ids) > max_seq_len - 1)

    if is_too_short or is_too_long:
        removed_count += 1
    else:
        filtered_data.append(item)

  print(f"Removed {removed_count} sentences due to length constraints (min raw tokens: {min_raw_tokens}, max effective seq_len for src: {max_seq_len-2}, for tgt: {max_seq_len-1}).")
  ds_raw = filtered_data # Update ds_raw to the filtered data

  # Keep 90% for training, 10% for validation
  train_ds_size = int(0.9 * len(ds_raw))
  val_ds_size = len(ds_raw) - train_ds_size
  train_ds_raw, val_ds_raw = random_split(ds_raw, [train_ds_size, val_ds_size])

  train_ds = BilingualDataset(train_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
  val_ds = BilingualDataset(val_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])

  # Find the maximum length of each sentence in the source and target sentence (after filtering)
  max_len_src = 0
  max_len_tgt = 0

  for item in ds_raw:
    src_ids = tokenizer_src.encode(item['translation'][config['lang_src']]).ids
    tgt_ids = tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids
    max_len_src = max(max_len_src, len(src_ids))
    max_len_tgt = max(max_len_tgt, len(tgt_ids))

  print(f'Max length of source sentence (after filtering): {max_len_src}')
  print(f'Max length of target sentence (after filtering): {max_len_tgt}')

  train_dataloader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
  val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=True)

  return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt

### `get_model` Function: Instantiating Our Transformer

This simple helper function creates an instance of our `Transformer` class, using the configuration parameters we've defined, such as model dimension (`d_model`), number of layers (`N`), and feed-forward dimension (`d_ff`).

In [ ]:
def get_model(config, vocab_src_len, vocab_tgt_len):
  model = build_transformer(vocab_src_len, vocab_tgt_len, config["seq_len"], config['seq_len'], d_model=config['d_model'], N=config['N'], num_heads=8, dropout=0.1, d_ff=config['d_ff'], verbose=False)
  return model

### `train_model` Function: The Learning Loop (Epochs)

This is the core training function. It loops through the dataset multiple times (each loop is an 'epoch'). In each epoch, it feeds batches of data to the model, calculates how far off the model's predictions are (the 'loss'), and then adjusts the model's internal weights to reduce that error. This iterative process is how the model learns to translate better and better.

In [ ]:
def get_weights_file_path(config, epoch: str):
  model_folder = f"{config['datasource']}_{config['model_folder']}"
  model_filename = f"{config['model_basename']}{epoch}.pt"
  return str(Path('.') / model_folder / model_filename)

In [ ]:
def latest_weights_file_path(config):
  model_folder = f"{config['datasource']}_{config['model_folder']}"
  model_filename = f"{config['model_basename']}*"
  weights_files = list(Path(model_folder).glob(model_filename))
  if len(weights_files) == 0:
      return None
  weights_files.sort()
  return str(weights_files[-1])

# Training Loop

We'll spend more time in following weeks discussing this - for now, we'll quickly walk through what's happening:

1. Configure the training device (GPU/CPU) and print details. Set device in PyTorch.

2. Create directory for saving model weights based on config.

3. Get data loaders, tokenizers, and model. Move model to configured device.

4. Initialize Adam optimizer with learning rate and epsilon from config.

5. Set up initial training parameters like start epoch and global step.

6. Define cross-entropy loss function with label smoothing, ignoring padding.

---

- Main training loop over epochs:

  - Clear cache, set model to train mode, initialize progress bar.

  - For each batch:

    - Move data to device, run model forward/backward passes.
    - Compute loss, backprop, update model weights.
    - Increment global step.
  - After each epoch, save model and optimizer checkpoint.

In [ ]:
import warnings
from tqdm import tqdm
import os
from pathlib import Path

def train_model(config):
  # Define the device
  device = "cuda" if torch.cuda.is_available() else "mps" if torch.has_mps or torch.backends.mps.is_available() else "cpu"
  print("Using device:", device)
  if (device == 'cuda'):
    print(f"Device name: {torch.cuda.get_device_name(device.index)}")
    print(f"Device memory: {torch.cuda.get_device_properties(device.index).total_memory / 1024 ** 3} GB")
  else:
    print("Please ensure you're in a GPU enabled Colab Notebook instance.")
  device = torch.device(device)

  # Make sure the weights folder exists
  Path(f"{config['datasource']}_{config['model_folder']}").mkdir(parents=True, exist_ok=True)

  train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
  model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

  optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], eps=1e-9)

  initial_epoch = 0
  global_step = 0

  loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id('[PAD]'), label_smoothing=0.1).to(device)

  for epoch in range(initial_epoch, config['num_epochs']):
    torch.cuda.empty_cache()
    model.train()
    batch_iterator = tqdm(train_dataloader, desc=f"Processing Epoch {epoch:02d}")
    for batch in batch_iterator:
      encoder_input = batch['encoder_input'].to(device)
      decoder_input = batch['decoder_input'].to(device)
      encoder_mask = batch['encoder_mask'].to(device)
      decoder_mask = batch['decoder_mask'].to(device)

      encoder_output = model.encode(encoder_input, encoder_mask)
      decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
      proj_output = model.project(decoder_output)

      label = batch['label'].to(device)

      loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))
      batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

      loss.backward()

      optimizer.step()
      optimizer.zero_grad(set_to_none=True)

      global_step += 1

    model_filename = get_weights_file_path(config, f"{epoch:02d}")
    torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'global_step': global_step
    }, model_filename)

## Part 5: Using the Trained Transformer (Putting the Brain to Work)

After training, our Transformer model has learned to translate. We can now load this trained 'brain' and ask it to translate new, unseen sentences.

### `load_model` and `generate` Functions: Translation in Action

These functions handle loading the saved model and performing inference (generating translations). The `generate` function takes an English sentence, feeds it to the encoder, and then uses the decoder to generate Italian words one by one until it predicts an 'end of sentence' token or reaches a maximum length. This is how the trained Transformer translates your input!

In [ ]:
config = {
  "batch_size": 64,
  "num_epochs": 6,
  "lr": 1e-4,
  "seq_len": 128,
  "d_model": 256, # Further reduced from 64
  "N": 3, # Further reduced from 2 layers
  "d_ff": 128, # Further reduced from 256
  "datasource": 'opus_books',
  "lang_src": "en",
  "lang_tgt": "it",
  "model_folder": "trained__en_it_translation_model",
  "model_basename": "encoder_decoder_model_"
}

### Prevent Memory Fragmentation

To prevent potential memory fragmentation issues during training, we can set the `PYTORCH_ALLOC_CONF` environment variable. This allows PyTorch to expand memory segments, which can be beneficial for long training runs or large models.

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
train_model(config)

Using device: cuda
Device name: Tesla T4
Device memory: 14.56317138671875 GB
Removed 7476 sentences due to length constraints (min raw tokens: 10, max effective seq_len for src: 126, for tgt: 127).
Max length of source sentence (after filtering): 126
Max length of target sentence (after filtering): 127


Processing Epoch 05: 100%|██████████| 350/350 [01:25<00:00,  4.11it/s, loss=5.614]


In [ ]:
def load_model(config):
    # Get dataloaders and tokenizers
    _, _, tokenizer_src, tokenizer_tgt = get_ds(config)

    # Initialize model
    model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size())

    # Load trained weights
    model_filename = latest_weights_file_path(config)
    if model_filename:
        print(f"Loading weights from {model_filename}")
        state = torch.load(model_filename)
        model.load_state_dict(state['model_state_dict'])

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    return model, tokenizer_src, tokenizer_tgt, device

def generate(model, tokenizer_src, tokenizer_tgt, src_text, device, max_length=350):
    model.eval()

    enc_input = tokenizer_src.encode(src_text).ids
    enc_input = torch.tensor([tokenizer_src.token_to_id('[SOS]')] + enc_input + [tokenizer_src.token_to_id('[EOS]')]).unsqueeze(0)

    enc_mask = (enc_input != tokenizer_src.token_to_id('[PAD]')).unsqueeze(0).unsqueeze(0).int()

    enc_input = enc_input.to(device)
    enc_mask = enc_mask.to(device)

    with torch.no_grad():
        enc_output = model.encode(enc_input, enc_mask)
        dec_input = torch.tensor([[tokenizer_tgt.token_to_id('[SOS]')]]).to(device)

        for _ in range(max_length):
            dec_mask = causal_mask(dec_input.size(1)).to(device)

            dec_output = model.decode(enc_output, enc_mask, dec_input, dec_mask)
            proj_output = model.project(dec_output)

            next_word = proj_output[:, -1].argmax(dim=-1)
            dec_input = torch.cat([dec_input, next_word.unsqueeze(-1)], dim=1)

            if next_word.item() == tokenizer_tgt.token_to_id('[EOS]') :
                break

    translated_tokens = [tokenizer_tgt.id_to_token(t.item()) for t in dec_input[0]]
    translated_text = ' '.join([t for t in translated_tokens if t not in ['[SOS]', '[EOS]', '[PAD]']])

    return translated_text


model, tokenizer_src, tokenizer_tgt, device = load_model(config)
model.eval()

Removed 7476 sentences due to length constraints (min raw tokens: 10, max effective seq_len for src: 126, for tgt: 127).
Max length of source sentence (after filtering): 126
Max length of target sentence (after filtering): 127
Loading weights from opus_books_trained__en_it_translation_model/encoder_decoder_model_05.pt


Transformer(
  (encoder): EncoderStack(
    (layers): ModuleList(
      (0-2): 3 x EncoderBlock(
        (self_attention_block): MultiHeadAttention(
          (w_q): Linear(in_features=256, out_features=256, bias=False)
          (w_k): Linear(in_features=256, out_features=256, bias=False)
          (w_v): Linear(in_features=256, out_features=256, bias=False)
          (w_o): Linear(in_features=256, out_features=256, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward_block): FeedForwardBlock(
          (linear_1): Linear(in_features=256, out_features=128, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=128, out_features=256, bias=True)
        )
        (residual_connections): ModuleList(
          (0-1): 2 x ResidualConnection(
            (dropout): Dropout(p=0.1, inplace=False)
            (layernorm): LayerNormalization()
          )
        )
      )
    )
    (norm): LayerNormalizat

## Conclusion

We've covered the journey of building a Transformer from its smallest components to a functional translation model. While the translations might not be perfect with a small model and limited training, the core architecture and process remain the same as in much larger, state-of-the-art models!

In [ ]:
test_sentences = [
        "the weather is beautiful today",
        "how are you?"
    ]

print("English to Italian Translations:")
print("-" * 50)
for sentence in test_sentences:
    # Limit the generation length to the model's configured sequence length
    translation = generate(model, tokenizer_src, tokenizer_tgt, sentence, device, max_length=config['seq_len'])
    print(f"EN: {sentence}")
    print(f"IT: {translation}")
    print("-" * 50)

English to Italian Translations:
--------------------------------------------------
EN: the weather is beautiful today
IT: E è un ' è un po ’ è un ' è un ' è un ' è un ' è un ' è un ' è un ' è ?
--------------------------------------------------
EN: how are you?
IT: E che è ? — è ? — ? — è ?
--------------------------------------------------


In [ ]:
tokenizer_src

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"[SOS]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"[EOS]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=None, pre_tokenizer=Whitespace(), post_processor=None, decoder=None, model=WordLevel(vocab={"[UNK]":0, "[PAD]":1, "[SOS]":2, "[EOS]":3, ",":4, "the":5, "and":6, ".":7, "to":8, "I":9, "of":10, "a":11, "'":12, "in":13, "was":14, "that":15, "he":16, "it":17, ";":18, "had":19, "his":20, "not":21, "with":22, "her":23, "you":24, "as":25, "for":26, "she":27, "my":28, "-":29, "at":30, "but":31, "him":32, "me":33, "is":34, """:35, "on":36, "be":37, ":

In [ ]:
tokenizer_tgt

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"[SOS]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"[EOS]", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=None, pre_tokenizer=Whitespace(), post_processor=None, decoder=None, model=WordLevel(vocab={"[UNK]":0, "[PAD]":1, "[SOS]":2, "[EOS]":3, ",":4, ".":5, "e":6, "di":7, "che":8, "—":9, "’":10, "la":11, "non":12, "a":13, "il":14, "un":15, "in":16, "per":17, "si":18, ";":19, "con":20, "una":21, "era":22, "le":23, "l":24, "mi":25, "ma":26, "è":27, "da":28, "'":29, "?":30, "del":31, "i":32, "come":33, "più":34, "della":35, "lo":36, "disse":37, "gli":

#### Acknowledgements

This notebook is adapted from the original work by **[AI Maker Space](https://github.com/AI-Maker-Space/)** — all credit for the core implementation goes to their team.

Additional resources this notebook draws from:

- https://blog.floydhub.com/the-transformer-in-pytorch/
- https://arxiv.org/pdf/1706.03762.pdf
- https://txt.cohere.com/what-are-transformer-models/
- https://jalammar.github.io/illustrated-transformer/